# Thai Election OCR Pipeline — Typhoon OCR 1.5
## Super AI Engineer Season 6

**Typhoon OCR 1.5-3B-QAT** — โมเดลที่สร้างมาเพื่อ Thai OCR โดยเฉพาะ
- เร็ว ~3-5 วินาที/หน้า (vs 42s ของ Qwen)
- แม่นกว่า Gemini 2.5 Pro บน Thai docs
- รันบน T4 GPU ได้ (4-bit quantized)

**Runtime: GPU (T4)** — `Runtime > Change runtime type > T4 GPU`

In [ ]:
# === Cell 1: Install Dependencies ===
!pip install -q git+https://github.com/huggingface/transformers accelerate
!pip install -q bitsandbytes qwen-vl-utils pillow
!pip install -q rapidfuzz tqdm pandas 2>/dev/null
print('Install complete.')

In [ ]:
# === Cell 2: Kaggle Download ===
import os, json

os.makedirs('/root/.kaggle', exist_ok=True)
from google.colab import userdata
username = userdata.get('KAGGLE_USERNAME')
key = userdata.get('KAGGLE_KEY')
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': username, 'key': key}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print(f'Kaggle: {username}')

COMPETITION = 'super-ai-engineer-season-6-ocr-2569'
!kaggle competitions download -c {COMPETITION} -p /content/ 2>&1

import glob as glob_module
for zf in glob_module.glob('/content/*.zip'):
    print(f'Extracting: {zf}')
    !unzip -qo "{zf}" -d /content/
print('Done.')

In [ ]:
# === Cell 3: Imports & Config ===
import os, re, json, time, gc
import pandas as pd
from pathlib import Path
from collections import defaultdict
from datetime import datetime
from tqdm.notebook import tqdm
from rapidfuzz import fuzz
from PIL import Image, ImageEnhance
import glob as glob_module

# Auto-discover paths
found = glob_module.glob('/content/**/images', recursive=True)
IMAGE_DIR = Path(found[0]) if found else Path('/content/data/images')
DATA_DIR = IMAGE_DIR.parent
SAMPLE_LABELS_DIR = DATA_DIR / 'sample_labels'

found_tmpl = glob_module.glob('/content/**/submission_template.csv', recursive=True)
SUBMISSION_TEMPLATE = Path(found_tmpl[0]) if found_tmpl else DATA_DIR / 'submission_template.csv'

OUTPUT_DIR = Path('/content/output')
OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT_FILE = OUTPUT_DIR / 'ocr_results.json'
PROGRESS_FILE = OUTPUT_DIR / 'progress.txt'

images = sorted(IMAGE_DIR.glob('*.png'))
print(f'Images: {len(images)}')
print(f'Template: {SUBMISSION_TEMPLATE} (exists={SUBMISSION_TEMPLATE.exists()})')
print(f'Sample labels: {SAMPLE_LABELS_DIR.exists()}')

In [ ]:
# === Cell 4: Load Template & Group Documents ===
template_df = pd.read_csv(SUBMISSION_TEMPLATE)
print(f'Template: {template_df.shape}')

def group_documents(image_dir):
    docs = defaultdict(list)
    pattern = re.compile(r'^(.+?)(?:_page(\d+))?\.png$')
    for p in sorted(image_dir.glob('*.png')):
        m = pattern.match(p.name)
        if m:
            doc_id = m.group(1)
            page = int(m.group(2)) if m.group(2) else 1
            docs[doc_id].append((page, str(p)))
    for d in docs:
        docs[d].sort()
    return dict(docs)

documents = group_documents(IMAGE_DIR)
print(f'Documents: {len(documents)}')

template_doc_rows = template_df.groupby('doc_id').size().to_dict()
doc_expected = defaultdict(list)
for _, row in template_df.iterrows():
    doc_expected[row['doc_id']].append({
        'id': row['id'],
        'row_num': row['row_num'],
        'party_name': row['party_name'],
    })
for d in doc_expected:
    doc_expected[d].sort(key=lambda x: x['row_num'])

doc_types = {d: ('partylist' if 'party_list' in d else 'constituency') for d in documents}
c = sum(1 for v in doc_types.values() if v == 'constituency')
p = sum(1 for v in doc_types.values() if v == 'partylist')
total_pages = sum(len(v) for v in documents.values())
print(f'Constituency: {c}, Party list: {p}, Total pages: {total_pages}')

In [ ]:
# === Cell 5: Load Typhoon OCR 1.5 ===
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

MODEL_ID = "scb10x/typhoon-ocr1.5-3b-qat"

print(f'Loading {MODEL_ID}...')
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,  # CRITICAL: must be bfloat16, not float16
    device_map="auto",
    trust_remote_code=True,
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

vram_used = torch.cuda.memory_allocated() / 1024**3
vram_total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'Model loaded! VRAM: {vram_used:.1f}/{vram_total:.1f} GB')

In [ ]:
# === Cell 6: OCR Functions (Typhoon-specific) ===
from PIL import Image, ImageEnhance

THAI_TO_ARABIC = str.maketrans('\u0e50\u0e51\u0e52\u0e53\u0e54\u0e55\u0e56\u0e57\u0e58\u0e59', '0123456789')

def clean_vote(text):
    s = str(text).translate(THAI_TO_ARABIC)
    s = re.sub(r'[,.\'\"\s]', '', s)
    s = re.sub(r'[^0-9]', '', s)
    return s if s else '0'

def resize_image(img, max_size=1800):
    """Resize to max 1800px (Typhoon was trained with this)."""
    width, height = img.size
    if width >= height and width > max_size:
        scale = max_size / float(width)
        img = img.resize((max_size, int(height * scale)), Image.LANCZOS)
    elif height > width and height > max_size:
        scale = max_size / float(height)
        img = img.resize((int(width * scale), max_size), Image.LANCZOS)
    return img

# CRITICAL: This is the EXACT prompt Typhoon OCR was trained with
TYPHOON_PROMPT = """Extract all text from the image.

Instructions:
- Only return the clean Markdown.
- Do not include any explanation or extra text.
- You must include all information on the page.

Formatting Rules:
- Tables: Render tables using <table>...</table> in clean HTML format.
- Equations: Render equations using LaTeX syntax with inline ($...$) and block ($$...$$).
- Images/Charts/Diagrams: Wrap visual areas in:

<figure>
Describe the image's main elements (people, objects, text), note contextual clues, mention visible text, provide deeper analysis, comment on style/architecture. Describe in Thai.
</figure>

- Page Numbers: Wrap in <page_number>...</page_number>
- Checkboxes: Use ☐ for unchecked and ☑ for checked boxes."""


def ocr_page_typhoon(img_path):
    """OCR a single page with Typhoon OCR using the EXACT trained prompt."""
    img = Image.open(img_path)
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img = resize_image(img, 1800)
    
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": TYPHOON_PROMPT}
        ]
    }]
    
    # CRITICAL: use return_dict=True with apply_chat_template
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)
    
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=2048,
            do_sample=False,
        )
    
    # Decode only new tokens
    out_ids = generated_ids[0][inputs['input_ids'].shape[1]:]
    response = processor.decode(out_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False).strip()
    
    del inputs, generated_ids
    torch.cuda.empty_cache()
    return response


def parse_html_table(response):
    """Parse HTML table output from Typhoon into party->votes dict."""
    pairs = {}
    
    # Extract rows from <tr>...</tr>
    import re
    rows = re.findall(r'<tr>(.*?)</tr>', response, re.DOTALL)
    
    for row in rows:
        cells = re.findall(r'<t[dh].*?>(.*?)</t[dh]>', row, re.DOTALL)
        cells = [c.strip() for c in cells]
        
        if not cells:
            continue
        
        # Skip header rows
        if any(kw in ' '.join(cells) for kw in ['หมายเลข', 'พรรค', 'คะแนน', 'ชื่อ', 'สังกัด', 'ลำดับ']):
            continue
        
        thai_cells = [c for c in cells if re.search(r'[\u0e00-\u0e7f]', c)]
        num_cells = [clean_vote(c) for c in cells if clean_vote(c) != '0']
        
        if thai_cells and num_cells:
            party = thai_cells[-1]
            vote = num_cells[-1]
            if len(party) >= 2:
                pairs[party] = vote
    
    # Fallback: try markdown table (pipe-delimited)
    if not pairs:
        for line in response.split('\n'):
            line = line.strip()
            if not line or '---' in line:
                continue
            if '|' in line:
                cells = [c.strip() for c in line.split('|')]
                cells = [c for c in cells if c]
                if any(kw in line for kw in ['หมายเลข', 'พรรค', 'คะแนน', 'ชื่อ', 'สังกัด']):
                    continue
                thai_cells = [c for c in cells if re.search(r'[\u0e00-\u0e7f]', c)]
                num_cells = [clean_vote(c) for c in cells if clean_vote(c) != '0']
                if thai_cells and num_cells:
                    party = thai_cells[-1]
                    vote = num_cells[-1]
                    if len(party) >= 2:
                        pairs[party] = vote
    
    return pairs


def extract_document(doc_id, page_paths, doc_type, expected_parties):
    all_pairs = {}
    for page_num, img_path in page_paths:
        try:
            response = ocr_page_typhoon(img_path)
            pairs = parse_html_table(response)
            all_pairs.update(pairs)
        except Exception as e:
            print(f'    Page {page_num} error: {e}')
            torch.cuda.empty_cache()
    
    results = []
    used = set()
    for exp in expected_parties:
        best = None
        best_score = 0
        for ocr_name, vote in all_pairs.items():
            if ocr_name in used:
                continue
            score = max(
                fuzz.ratio(exp, ocr_name),
                fuzz.partial_ratio(exp, ocr_name),
                fuzz.token_sort_ratio(exp, ocr_name),
            )
            if score > best_score and score >= 50:
                best_score = score
                best = (ocr_name, vote)
        if best:
            used.add(best[0])
            results.append({'party_name': exp, 'votes': best[1], 'ocr_name': best[0], 'score': best_score})
        else:
            results.append({'party_name': exp, 'votes': '0', 'ocr_name': '', 'score': 0})
    return results

print('Typhoon OCR functions ready (with correct prompt + HTML table parser).')

In [ ]:
# === Cell 7: Test on 1 document ===
test_doc = list(documents.keys())[0]
test_type = doc_types[test_doc]
expected = [e['party_name'] for e in doc_expected[test_doc]]

print(f'Testing: {test_doc} ({test_type})')
print(f'Pages: {len(documents[test_doc])}, Parties: {len(expected)}')

# First, show raw OCR output (page by page)
print('\n--- Raw OCR Output (page by page) ---')
for pn, pp in documents[test_doc]:
    t0 = time.time()
    raw = ocr_page_typhoon(pp)
    dt = time.time() - t0
    print(f'\nPage {pn} ({dt:.1f}s):')
    print(raw[:1500])
    if len(raw) > 1500:
        print(f'... (total {len(raw)} chars)')

# Now extract with matching
print('\n--- Extraction ---')
t0 = time.time()
results_test = extract_document(test_doc, documents[test_doc], test_type, expected)
elapsed = time.time() - t0

matched = sum(1 for r in results_test if r['votes'] != '0')
print(f'Matched: {matched}/{len(expected)} in {elapsed:.1f}s')
for r in results_test:
    m = '✓' if r['score'] >= 70 else '?' if r['score'] >= 50 else '✗'
    ocr_name = r.get('ocr_name', '')[:25]
    print(f"  {m} {r['party_name']:<25s} -> {r['votes']:>8s}  (score={r['score']:3.0f} ocr='{ocr_name}')")

# Ground truth comparison
label_file = SAMPLE_LABELS_DIR / f'{test_doc}.json'
if label_file.exists():
    with open(label_file, encoding='utf-8') as f:
        label = json.load(f)
    label_map = {r.get('party', r.get('party_name', '')): str(r['votes']) for r in label.get('results', [])}
    print(f'\n--- Ground Truth ---')
    for r in results_test:
        actual = next((v for k, v in label_map.items() if fuzz.ratio(r['party_name'], k) > 70), '?')
        ok = '✓' if actual == r['votes'] else '✗'
        print(f"  {ok} {r['party_name']:<25s}  actual={actual:>8s}  pred={r['votes']:>8s}")

per_page = elapsed / len(documents[test_doc])
print(f'\nSpeed: {per_page:.1f}s/page x {total_pages} pages = ~{per_page*total_pages/60:.0f} min total')

In [ ]:
# === Cell 8: Checkpoint & Save ===

def load_checkpoint():
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, encoding='utf-8') as f:
            return json.load(f)
    return {}

def save_checkpoint(results):
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

def save_submission_csv(results, template_df, doc_expected, output_dir):
    votes = {}
    for doc_id, rows in doc_expected.items():
        ocr = results.get(doc_id, [])
        for i, exp in enumerate(rows):
            v = ocr[i].get('votes', '0') if i < len(ocr) else '0'
            votes[exp['id']] = v if v else '0'
    sub = template_df[['id']].copy()
    sub['votes'] = sub['id'].map(votes).fillna('0')
    path = output_dir / 'submission.csv'
    sub.to_csv(path, index=False)
    return path

def update_progress(done, total, errors, start_time):
    elapsed = time.time() - start_time
    speed = done / elapsed if elapsed > 0 else 0
    eta = (total - done) / speed if speed > 0 else 0
    with open(PROGRESS_FILE, 'w') as f:
        f.write(f'Completed: {done}/{total} ({done/total*100:.1f}%)\n')
        f.write(f'Errors: {len(errors)} | Elapsed: {elapsed/60:.1f}m | ETA: {eta/60:.1f}m\n')
        f.write(f'Updated: {datetime.now().strftime("%H:%M:%S")}\n')

print('Save functions ready.')

In [ ]:
# === Cell 9: Main Processing Loop ===
results = load_checkpoint()
print(f'Loaded {len(results)} existing results.')

doc_ids_sorted = sorted(documents.keys(), key=lambda d: (doc_types.get(d, 'z'), d))
todo = [d for d in doc_ids_sorted if d not in results]
print(f'To process: {len(todo)} (skipped {len(results)})')

SAVE_EVERY = 5
errors = []
t_start = time.time()

for i, doc_id in enumerate(tqdm(todo, desc='OCR')):
    doc_type = doc_types[doc_id]
    expected_parties = [e['party_name'] for e in doc_expected[doc_id]]
    
    try:
        rows = extract_document(doc_id, documents[doc_id], doc_type, expected_parties)
        results[doc_id] = rows
        
        matched = sum(1 for r in rows if r.get('votes', '0') != '0')
        if matched < len(expected_parties) * 0.3:
            print(f'  WARN {doc_id}: {matched}/{len(expected_parties)}')
    except Exception as e:
        errors.append(doc_id)
        print(f'  ERROR {doc_id}: {e}')
        results[doc_id] = [{'party_name': p, 'votes': '0'} for p in expected_parties]
        torch.cuda.empty_cache()
    
    done = len(results)
    if done % SAVE_EVERY == 0 or i == len(todo) - 1:
        save_checkpoint(results)
        save_submission_csv(results, template_df, doc_expected, OUTPUT_DIR)
        update_progress(done, len(documents), errors, t_start)
        el = time.time() - t_start
        spd = el / done
        eta = spd * (len(todo) - i - 1)
        print(f'  [{done}/{len(documents)}] {el/60:.1f}m, ETA {eta/60:.0f}m')

save_checkpoint(results)
save_submission_csv(results, template_df, doc_expected, OUTPUT_DIR)
el = time.time() - t_start
print(f'\nDone! {len(results)}/{len(documents)} in {el/60:.1f} min')
if errors: print(f'Errors: {errors}')
print(f'CSV: {OUTPUT_DIR}/submission.csv')

In [ ]:
# === Cell 10: Validate ===
def levenshtein(s1, s2):
    if len(s1) < len(s2): return levenshtein(s2, s1)
    if len(s2) == 0: return len(s1)
    prev = list(range(len(s2) + 1))
    for i, c1 in enumerate(s1):
        curr = [i + 1]
        for j, c2 in enumerate(s2):
            curr.append(min(curr[j]+1, prev[j+1]+1, prev[j]+(c1!=c2)))
        prev = curr
    return prev[-1]

results = load_checkpoint()
if SAMPLE_LABELS_DIR.exists():
    total_dist = total_pairs = 0
    for jf in sorted(SAMPLE_LABELS_DIR.glob('*.json')):
        with open(jf, encoding='utf-8') as f:
            label = json.load(f)
        doc_id = jf.stem
        expected_rows = doc_expected.get(doc_id, [])
        ocr_rows = results.get(doc_id, [])
        print(f'\n=== {doc_id} ===')
        for lr in label.get('results', []):
            actual = str(lr['votes'])
            party = lr.get('party', lr.get('party_name', ''))
            pred = '0'
            for i, exp in enumerate(expected_rows):
                if fuzz.ratio(party, exp['party_name']) > 70 and i < len(ocr_rows):
                    pred = ocr_rows[i].get('votes', '0')
                    break
            dist = levenshtein(actual, pred)
            total_dist += dist
            total_pairs += 1
            if dist > 0:
                print(f'  {party:30s}  actual={actual:>8s}  pred={pred:>8s}  dist={dist}')
    print(f'\nMean Levenshtein: {total_dist/total_pairs:.4f} ({total_pairs} pairs)')

In [ ]:
# === Cell 11: Download ===
from google.colab import files
files.download(str(OUTPUT_DIR / 'submission.csv'))